<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">


## 👋 Bem-vindo ao seu primeiro Notebook!

### O que é um Notebook?
Um **notebook** é como um documento interativo onde você pode escrever código e executá-lo passo a passo. Diferente de um script tradicional, aqui você pode:
- Escrever código em **células** independentes
- **Executar cada célula** individualmente (clicando no botão ▶️ ou pressionando `Shift + Enter`)
- Ver os **resultados** imediatamente abaixo de cada célula
- Combinar código com **textos explicativos** (como este que você está lendo!)

### Como usar este Notebook?
1. **Leia as explicações** nas células de texto (fundo branco)
2. **Execute as células de código** (fundo cinza) na ordem, de cima para baixo
3. **Observe os resultados** que aparecem logo abaixo de cada célula executada

> ⚠️ **Importante:** Sempre execute as células na ordem! Algumas células dependem de variáveis criadas nas células anteriores.

### O que este laboratório faz?
Neste lab, vamos aprender a **importar dados de arquivos CSV** (um formato comum de planilhas) e salvá-los como **tabelas Delta** no Databricks. Isso é o primeiro passo em qualquer projeto de dados: trazer os dados brutos para dentro da plataforma.

### 🔌 Antes de começar: Conectando ao Cluster

Para executar código no Databricks, você precisa estar conectado a um **cluster**. Mas o que é isso?

> 💡 **O que é um Cluster?** Um cluster é um conjunto de computadores (máquinas) que trabalham juntos para processar seus dados. Pense nele como o "motor" que dá força ao seu notebook — sem ele, o código não roda.

#### Como conectar ao cluster:

| Passo | O que fazer |
|-------|------------|
| 1️⃣ | No canto superior direito do notebook, localize a bolinha verde que indica se o cluster está ligado (verde total) ou não conectado (verde contornado) |
| 2️⃣ | Clique nela e uma lista de clusters disponíveis vai aparecer |
| 3️⃣ | Selecione o cluster **`cluster_dbw_treinamento_dtbks`**. Caso não apareça, clique em **"More"** |
| 4️⃣ | Aguarde até o status mudar para **conectado** (indicador verde) |

> ⚠️ **Importante:** Se o cluster estiver desligado, ele pode levar alguns minutos para iniciar. Aguarde até aparecer o indicador verde antes de executar qualquer célula.

> ℹ️ **Dica:** Você só precisa conectar uma vez! Depois de conectado, todas as células do notebook usarão o mesmo cluster automaticamente.

### Referências
* [Leitura de Arquivos CSV](https://learn.microsoft.com/pt-br/azure/databricks/external-data/csv)
* [Notebook Exemplo - CSV](https://docs.databricks.com/_extras/notebooks/source/read-csv-files.html)
* [Salvando uma Tabela DELTA](https://docs.databricks.com/delta/tutorial.html#create-a-table)


### ⚙️ Parâmetros Iniciais

Antes de começar a trabalhar com dados, precisamos definir algumas **configurações básicas**. É como preencher um formulário antes de começar:

- **`url`** → O caminho (endereço) onde estão os arquivos CSV que vamos importar
- **`catalog_name`** → O nome do "catálogo" onde as tabelas serão salvas (pense como uma pasta principal)
- **`schema_name`** → O nome do "schema" (pense como uma subpasta dentro do catálogo)

> 💡 **Dica:** Usamos variáveis para guardar esses valores porque eles são reutilizados várias vezes ao longo do notebook. Se precisar mudar algo, basta alterar aqui em cima!

> ✏️ **Ação necessária:** Na célula abaixo, você precisa preencher a variável `url` e a variável `schema_name`. Siga as instruções abaixo.

---

#### 📌 Como obter o caminho (`url`) da pasta de dados:

O caminho da pasta **não é simplesmente o seu e-mail**. Você precisa copiar o caminho exato pelo próprio Databricks. Siga estes passos:

1. No menu lateral esquerdo, clique em **Workspace** (ícone de pasta no menu de apoio)
2. Navegue até a pasta onde estão os arquivos CSV do laboratório (ex: `lab_irb_2025/dados/`)
3. Clique com o **botão direito** na pasta `dados`
4. Selecione **"Copy URL/Path" -> Full path** 
5. Cole o valor na variável `url` na célula abaixo

> ⚠️ **Atenção:** O caminho copiado deve começar com `/Workspace/Users/...`. Certifique-se de adicionar uma **barra `/` no final** do caminho (ex: `.../dados/`).

#### 👤 Como preencher o `schema_name`:

Coloque seu **nome e sobrenome** separados por underline (`_`), sem acentos e em minúsculo.

Exemplos: `maria_silva`, `joao_santos`, `ana_oliveira`

In [0]:

# Importando bibliotecas necessárias
import pandas as pd                      # Pandas: biblioteca para ler arquivos CSV
from pyspark.sql import SparkSession     # SparkSession: motor de processamento de dados do Databricks

# ============================================================
# ⚠️ ALTERE AS VARIÁVEIS ABAIXO COM SEUS DADOS PESSOAIS
# ============================================================

# Caminho onde estão os arquivos CSV
# → COLE AQUI o caminho que você copiou do Workspace (veja instruções acima)
# → Lembre-se de manter a barra / no final!
url = f"<sua_url_aqui>/"

# Nome do catálogo (não altere, a menos que instruído)
catalog_name = f"prd_treinamento_databricks"

# Nome do schema (coloque SEU NOME aqui, sem acentos, com underline)
# Exemplos: "maria_silva", "joao_santos", "ana_oliveira"
schema_name  = f"<nome_sobrenome>"

### 📂 Criando o espaço para guardar os dados

Antes de importar os dados, precisamos criar o **local** onde eles serão armazenados. No Databricks, usamos o **Unity Catalog** para organizar dados de forma hierárquica:

```
Catalog (pasta principal)
  └── Schema (subpasta)
        └── Tabela (os dados propriamente ditos)
        └── outros recursos (Volumes, Modelos de IA, etc)
```

A célula abaixo faz duas coisas:
1. **Seleciona o catálogo** que vamos usar (`USE CATALOG`), usando o nome que definimos acima.
2. **Cria o schema** caso ele ainda não exista (`CREATE SCHEMA IF NOT EXISTS`), usando o seu `nome_sobrenome` informado acima.

> 💡 O comando `IF NOT EXISTS` é uma boa prática: ele só cria se ainda não existir, evitando erros ao rodar o notebook mais de uma vez.

In [0]:
# Passo 1: Selecionar o catálogo onde vamos trabalhar
create_catalog = f"USE CATALOG {catalog_name}"
spark.sql(create_catalog)
print(f"✅ Catálogo '{catalog_name}' selecionado com sucesso!")

# Passo 2: Criar o schema (se ainda não existir)
create_schema = f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}"
spark.sql(create_schema)
print(f"✅ Schema '{schema_name}' pronto para uso!")



✅ Catálogo 'catalog_robledo_uuch27' selecionado com sucesso!
✅ Schema 'bruna_robledo_teste' pronto para uso!


### 📥 Importando os arquivos CSV e salvando como tabelas Delta

Agora vem a parte principal! Vamos importar **6 arquivos CSV** e salvar cada um como uma **tabela Delta**.

#### O que é um arquivo CSV?
CSV (Comma-Separated Values) é um formato simples de arquivo onde os dados são separados por vírgulas. É como uma planilha do Excel, mas em formato texto puro.

#### O que é uma tabela Delta?
Delta é o formato otimizado do Databricks para armazenar dados. Ele é mais rápido, confiável e permite operações avançadas como versionamento (voltar no tempo nos dados).

#### O que é o Apache Spark?
O **Spark** é o "motor" por trás do Databricks. Imagine que você precisa processar milhões de linhas de dados — seu computador sozinho demoraria muito. O Spark resolve isso **distribuindo o trabalho entre vários computadores ao mesmo tempo**, como uma equipe onde cada pessoa cuida de uma parte da tarefa.

Quando convertemos dados para um **Spark DataFrame**, estamos dizendo: "Spark, pegue esses dados e prepare-os para serem processados de forma rápida e distribuída". É por isso que não salvamos diretamente do Pandas — o Pandas funciona apenas em um computador, enquanto o Spark pode usar o poder de vários.

> 🎯 **Resumindo:** Pandas = lê o arquivo | Spark = processa e salva com alta performance

#### O padrão de cada célula abaixo segue 3 passos:

```
1️⃣ pd.read_csv(...)           → Lê o arquivo CSV para a memória (usando a biblioteca de Python Pandas)
2️⃣ spark.createDataFrame(...) → Converte os dados para o formato Spark (necessário para salvar como tabela)
3️⃣ .saveAsTable(...)          → Salva os dados como uma tabela Delta no Unity Catalog
```

> 💡 **Por que `mode("overwrite")`?** Isso significa que, se a tabela já existir, ela será substituída pelos novos dados. Assim você pode rodar este notebook várias vezes sem erro.

---

#### Tabelas que serão criadas:
| # | Tabela | Descrição |
|---|--------|----------|
| 1 | `faturamento` | Dados de faturamento |
| 2 | `cnae` | Códigos de Classificação Nacional de Atividades Econômicas |
| 3 | `empresas_sp` | Cadastro de empresas de São Paulo |
| 4 | `ibge_senso` | Dados do Censo IBGE 2010 |
| 5 | `municipios` | Lista de municípios brasileiros |
| 6 | `naturezas` | Naturezas jurídicas das empresas |

In [0]:

# Nome da entidade (tabela) que vamos criar
entity_name  = f"faturamento"

# Monta os caminhos automaticamente usando as variáveis definidas lá em cima
table_name = f"{catalog_name}.{schema_name}.{entity_name}"  # ex: dev_treinamento_databricks.seu_nome.faturamento
file_name  = f"{url}{entity_name}.csv"                      # ex: /Workspace/.../dados/faturamento.csv

# === Os 3 passos da carga ===
df   = pd.read_csv(file_name)                          # 1️⃣ Lê o arquivo CSV com Pandas
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark DataFrame
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como tabela Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.faturamento' criada com 5629 registros!


In [0]:

entity_name = f"cnae"

table_name = f"{catalog_name}.{schema_name}.{entity_name}"
file_name  = f"{url}{entity_name}.csv"

df   = pd.read_csv(file_name)                          # 1️⃣ Lê o CSV
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.cnae' criada com 1359 registros!


In [0]:

entity_name = f"empresas_sp"

table_name = f"{catalog_name}.{schema_name}.{entity_name}"
file_name  = f"{url}{entity_name}.csv"

df   = pd.read_csv(file_name)                          # 1️⃣ Lê o CSV
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.empresas_sp' criada com 111534 registros!


In [0]:

entity_name = f"ibge_senso"

table_name = f"{catalog_name}.{schema_name}.{entity_name}"
file_name  = f"{url}{entity_name}.csv"

df   = pd.read_csv(file_name)                          # 1️⃣ Lê o CSV
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.ibge_senso' criada com 5565 registros!


In [0]:

entity_name = f"municipios"

table_name = f"{catalog_name}.{schema_name}.{entity_name}"
file_name  = f"{url}{entity_name}.csv"

df   = pd.read_csv(file_name)                          # 1️⃣ Lê o CSV
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.municipios' criada com 5571 registros!


In [0]:

entity_name = f"naturezas"

table_name = f"{catalog_name}.{schema_name}.{entity_name}"
file_name  = f"{url}{entity_name}.csv"

df   = pd.read_csv(file_name)                          # 1️⃣ Lê o CSV
s_df = spark.createDataFrame(df)                       # 2️⃣ Converte para Spark
s_df.write.mode("overwrite").saveAsTable(table_name)   # 3️⃣ Salva como Delta

print(f"✅ Tabela '{table_name}' criada com {s_df.count()} registros!")

✅ Tabela 'catalog_robledo_uuch27.bruna_robledo_teste.naturezas' criada com 90 registros!


### ✅ Pronto! Laboratório concluído!

Se todas as células acima executaram sem erros, **parabéns!** Você acabou de:

1. ✅ Configurar o ambiente (catálogo e schema)
2. ✅ Importar 6 arquivos CSV
3. ✅ Salvar todos como tabelas Delta no Unity Catalog

#### O que você pode fazer agora?
- Navegar até o **Catalog** (menu lateral esquerdo) para ver suas tabelas criadas
- Clicar em uma tabela para ver os dados, colunas e estatísticas

#### Próximos passos
Nos próximos laboratórios, vamos aprender a **consultar** esses dados, criar **análises** e **visualizações**!

---
> 📚 **Vocabulário aprendido neste lab:**
> - **Notebook** → Documento interativo para escrever e executar código
> - **Célula** → Bloco individual de código ou texto dentro do notebook
> - **CSV** → Formato de arquivo de dados separados por vírgula
> - **Delta** → Formato otimizado de armazenamento de dados do Databricks
> - **Unity Catalog** → Sistema de organização de dados (Catalog > Schema > Tabela)
> - **DataFrame** → Estrutura que representa dados em formato de tabela na memória